# JEPA thesis — Kaggle UI runner

This notebook embeds every file in `src/` and `configs/` as its own cell so you can:

1. **See** all the code in the Kaggle UI.
2. **Edit** any cell in-place and re-run just that cell to update the file.
3. **Run** the cells top-to-bottom, or one at a time to debug.

**First time:** right sidebar → Accelerator → **GPU T4 x2**, Internet → **On**, then *Save & Run All*.

**Re-running locally?** Regenerate this notebook with `python3 scripts/build_notebook.py` after editing `src/`.


In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"
REWRITE_FROM_NOTEBOOK = True  # if False, skip the %%writefile cells below


In [ ]:
import os, subprocess

def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd}")
    return r

net = subprocess.run(f"git ls-remote {REPO_URL}", shell=True, capture_output=True, text=True)
if net.returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Internet ON.")

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)


In [ ]:
sh(f'pip install -q -r {WORKDIR}/requirements.txt')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, _ = torch.cuda.get_device_capability(0)
    print(f'Device: {name}  (sm_{major}0)')
    if major < 7:
        raise RuntimeError(
            f'GPU {name} (sm_{major}0) is too old for the installed PyTorch. '
            'Right sidebar -> Accelerator -> GPU T4 x2.'
        )
else:
    print('Device: CPU (no GPU)')


In [ ]:
import os
from pathlib import Path

print('=== repo tree ===')
for root, dirs, files in os.walk('.'):
    if '.git' in root or '__pycache__' in root:
        continue
    for f in sorted(files):
        print(os.path.join(root, f))

print()
print('=== /kaggle/input tree (datasets) ===')
input_root = Path('/kaggle/input')
if input_root.exists():
    for p in sorted(input_root.rglob('*')):
        if p.is_dir():
            n_imgs = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.jpeg'))) + len(list(p.glob('*.png')))
            if n_imgs:
                print(f'  {p}   ({n_imgs} images)')
else:
    print('  (not on Kaggle)')


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""Central config. Detects whether we're running on Kaggle vs locally
so paths and device selection just work in both places."""
from __future__ import annotations

import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Tuple


def on_kaggle() -> bool:
    return os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def pick_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            try:
                major, _ = torch.cuda.get_device_capability(0)
                supported = torch.cuda.get_arch_list()
                if any(int(a.split("_")[1]) // 10 <= major for a in supported if a.startswith("sm_")):
                    torch.zeros(1, device="cuda")
                    return "cuda"
                print(f"[warn] GPU sm_{major}0 not supported by this torch "
                      f"({supported}); falling back to CPU. Pick a T4 GPU on Kaggle.")
            except Exception as e:
                print(f"[warn] CUDA present but unusable ({e}); falling back to CPU.")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except ImportError:
        pass
    return "cpu"


def find_isic_image_dir() -> Path:
    if not on_kaggle():
        return Path("data/ISIC2018_Task1-2_Training_Input")
    input_root = Path("/kaggle/input")
    print(f"[data] scanning {input_root} ...")
    if not input_root.exists():
        print(f"[data] ERROR: {input_root} does not exist")
        return Path("data/ISIC2018_Task1-2_Training_Input")
    top = sorted(input_root.iterdir())
    print(f"[data] top-level entries: {[p.name for p in top]}")
    image_exts = ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png", "*.PNG")
    def has_imgs(d: Path) -> bool:
        return any(d.glob(e) for e in image_exts)
    candidates = [
        input_root / "isic2018-challenge-task1-data-segmentation" / "ISIC2018_Task1-2_Training_Input",
        input_root / "isic2018-challenge-task1-data-segmentation",
    ]
    for c in candidates:
        exists = c.exists()
        imgs = sum(len(list(c.glob(e))) for e in image_exts) if exists else 0
        print(f"[data] candidate {c}  exists={exists}  imgs={imgs}")
        if exists and imgs:
            print(f"[data] using {c}")
            return c
    print("[data] candidates empty, scanning recursively ...")
    matches: list[tuple[Path, int]] = []
    for sub in input_root.rglob("*"):
        if not sub.is_dir():
            continue
        n = sum(len(list(sub.glob(e))) for e in image_exts)
        if n:
            matches.append((sub, n))
    matches.sort(key=lambda x: -x[1])
    for p, n in matches[:10]:
        print(f"[data]   found: {p}  ({n} images)")
    if matches:
        print(f"[data] using {matches[0][0]}")
        return matches[0][0]
    raise FileNotFoundError(
        f"No images found under {input_root}. Attach the ISIC dataset via "
        "kernel-metadata.json -> dataset_sources."
    )


@dataclass
class JEPACfg:
    img_size: int = 96
    patch_size: int = 8
    enc_dim: int = 192
    enc_depth: int = 12
    enc_heads: int = 3
    enc_mlp_ratio: float = 4.0
    pred_dim: int = 192
    pred_depth: int = 6
    pred_heads: int = 4
    pred_mlp_ratio: float = 4.0
    mask_scale: Tuple[float, float] = (0.15, 0.20)
    mask_aspect: Tuple[float, float] = (0.75, 1.5)
    ema_momentum: float = 0.996
    enc_lr: float = 1e-3
    pred_lr: float = 1e-3
    weight_decay: float = 0.05
    batch_size: int = 64
    epochs: int = 1
    save_every_epochs: int = 10

    @property
    def n_h(self) -> int:
        return self.img_size // self.patch_size

    @property
    def n_w(self) -> int:
        return self.img_size // self.patch_size


def _load_yaml_into_jepa(jepa: JEPACfg) -> None:
    cfg_path = Path(__file__).resolve().parent.parent / "configs" / "pretrain_jepa.yaml"
    if not cfg_path.exists():
        return
    try:
        import yaml
    except ImportError:
        print("[warn] pyyaml not installed; skipping configs/pretrain_jepa.yaml")
        return
    with open(cfg_path) as f:
        data = yaml.safe_load(f) or {}
    for k, v in data.items():
        if hasattr(jepa, k):
            setattr(jepa, k, v)


@dataclass
class Config:
    output_dir: Path = field(
        default_factory=lambda: Path("/kaggle/working") if on_kaggle() else Path("outputs")
    )
    data_dir: Path = field(default_factory=find_isic_image_dir)
    device: str = field(default_factory=pick_device)
    seed: int = 42
    jepa: JEPACfg = field(default_factory=JEPACfg)

    def __post_init__(self) -> None:
        self.output_dir.mkdir(parents=True, exist_ok=True)
        _load_yaml_into_jepa(self.jepa)


CONFIG = Config()


In [ ]:
%%writefile src/data.py
from __future__ import annotations

from pathlib import Path
from typing import Sequence

import torch
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T


class ISICImageDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        img_size: int = 96,
        augment: bool = False,
        exts: Sequence[str] = (".jpg", ".jpeg", ".png", ".bmp"),
    ):
        self.root = Path(root)
        if not self.root.exists():
            raise FileNotFoundError(f"ISIC image dir not found: {self.root}")
        exts_lower = {e.lower() for e in exts}
        self.paths = sorted(p for p in self.root.iterdir() if p.suffix.lower() in exts_lower)
        if not self.paths:
            raise RuntimeError(
                f"No images with ext {sorted(exts_lower)} found under {self.root}"
            )
        self.img_size = img_size
        self.augment = augment

        tfms: list = [
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BILINEAR),
        ]
        if augment:
            tfms.append(T.RandomHorizontalFlip())
        tfms += [
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ]
        self.transform = T.Compose(tfms)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img)


In [ ]:
%%writefile src/utils/__init__.py


In [ ]:
%%writefile src/utils/masking.py
from __future__ import annotations

from typing import Tuple

import torch


def sample_target_block(
    n_h: int,
    n_w: int,
    scale_range: Tuple[float, float] = (0.15, 0.20),
    aspect_range: Tuple[float, float] = (0.75, 1.5),
    generator: torch.Generator | None = None,
) -> dict:
    n_total = n_h * n_w
    s = torch.empty(1).uniform_(*scale_range, generator=generator).item()
    r = torch.empty(1).uniform_(*aspect_range, generator=generator).item()

    area = max(1.0, s * n_total)
    h_t = int(round((area * r) ** 0.5))
    w_t = int(round((area / max(r, 1e-6)) ** 0.5))
    h_t = max(1, min(h_t, n_h))
    w_t = max(1, min(w_t, n_w))

    top = int(torch.randint(0, n_h - h_t + 1, (1,), generator=generator).item())
    left = int(torch.randint(0, n_w - w_t + 1, (1,), generator=generator).item())

    mask_hw = torch.zeros(n_h, n_w, dtype=torch.bool)
    mask_hw[top : top + h_t, left : left + w_t] = True
    mask_flat = mask_hw.view(-1)

    tgt_indices = torch.nonzero(mask_flat, as_tuple=False).squeeze(-1)
    ctx_indices = torch.nonzero(~mask_flat, as_tuple=False).squeeze(-1)

    return {
        "tgt_mask_hw": mask_hw,
        "tgt_indices": tgt_indices,
        "ctx_indices": ctx_indices,
        "n_ctx": int(ctx_indices.shape[0]),
        "n_tgt": int(tgt_indices.shape[0]),
    }


In [ ]:
%%writefile src/model/__init__.py


In [ ]:
%%writefile src/model/vit.py
from __future__ import annotations

import torch
import torch.nn as nn


class PatchEmbed(nn.Module):
    def __init__(self, patch_size: int, in_chans: int, dim: int):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x).flatten(2).transpose(1, 2)


class Mlp(nn.Module):
    def __init__(self, dim: int, mlp_ratio: float, dropout: float = 0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Block(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_ratio: float, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, mlp_ratio, dropout)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


class ViT(nn.Module):
    def __init__(
        self,
        img_size: int = 96,
        patch_size: int = 8,
        in_chans: int = 3,
        dim: int = 192,
        depth: int = 12,
        heads: int = 3,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        if img_size % patch_size != 0:
            raise ValueError(f"img_size={img_size} not divisible by patch_size={patch_size}")
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_h = img_size // patch_size
        self.n_w = img_size // patch_size
        self.n_patches = self.n_h * self.n_w
        self.dim = dim

        self.patch_embed = PatchEmbed(patch_size, in_chans, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList(
            [Block(dim, heads, mlp_ratio, dropout) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

    def forward_context(
        self,
        x: torch.Tensor,
        ctx_indices: torch.Tensor,
        ctx_valid: torch.Tensor,
    ) -> torch.Tensor:
        all_patches = self.patch_embed(x) + self.pos_embed
        ctx = torch.gather(
            all_patches,
            1,
            ctx_indices.unsqueeze(-1).expand(-1, -1, self.dim),
        )
        kpm = ~ctx_valid
        for block in self.blocks:
            ctx = block(ctx, key_padding_mask=kpm)
        return self.norm(ctx)


In [ ]:
%%writefile src/model/predictor.py
from __future__ import annotations

import torch
import torch.nn as nn

from .vit import Block


class Predictor(nn.Module):
    def __init__(
        self,
        n_patches: int,
        dim: int = 192,
        depth: int = 6,
        heads: int = 4,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_patches = n_patches
        self.dim = dim

        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList(
            [Block(dim, heads, mlp_ratio, dropout) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(
        self,
        context_emb: torch.Tensor,
        ctx_valid: torch.Tensor,
        tgt_indices: torch.Tensor,
        tgt_valid: torch.Tensor,
    ) -> torch.Tensor:
        B = context_emb.shape[0]
        n_tgt = tgt_indices.shape[1]

        tgt_pos = self.pos_embed.expand(B, -1, -1).gather(
            1, tgt_indices.unsqueeze(-1).expand(-1, -1, self.dim)
        )
        mask_tokens = self.mask_token.expand(B, n_tgt, -1) + tgt_pos

        valid = torch.cat([ctx_valid, tgt_valid], dim=1)
        seq = torch.cat([context_emb, mask_tokens], dim=1)
        kpm = ~valid

        for block in self.blocks:
            seq = block(seq, key_padding_mask=kpm)
        seq = self.norm(seq)

        n_ctx = context_emb.shape[1]
        return seq[:, n_ctx:]


In [ ]:
%%writefile src/model/jepa.py
from __future__ import annotations

import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

from .predictor import Predictor
from .vit import ViT


class IJEPA(nn.Module):
    def __init__(
        self,
        img_size: int = 96,
        patch_size: int = 8,
        enc_dim: int = 192,
        enc_depth: int = 12,
        enc_heads: int = 3,
        enc_mlp_ratio: float = 4.0,
        pred_dim: int = 192,
        pred_depth: int = 6,
        pred_heads: int = 4,
        pred_mlp_ratio: float = 4.0,
        ema_momentum: float = 0.996,
    ):
        super().__init__()
        if enc_dim != pred_dim:
            raise ValueError(
                f"enc_dim ({enc_dim}) must equal pred_dim ({pred_dim}); "
                "the scaffold assumes shared embedding space."
            )
        self.ema_momentum = ema_momentum

        self.context_enc = ViT(
            img_size=img_size,
            patch_size=patch_size,
            dim=enc_dim,
            depth=enc_depth,
            heads=enc_heads,
            mlp_ratio=enc_mlp_ratio,
        )
        self.target_enc = copy.deepcopy(self.context_enc)
        for p in self.target_enc.parameters():
            p.requires_grad = False

        self.predictor = Predictor(
            n_patches=self.context_enc.n_patches,
            dim=pred_dim,
            depth=pred_depth,
            heads=pred_heads,
            mlp_ratio=pred_mlp_ratio,
        )

    @torch.no_grad()
    def update_target_encoder(self) -> None:
        m = self.ema_momentum
        for ctx_p, tgt_p in zip(self.context_enc.parameters(), self.target_enc.parameters()):
            tgt_p.data.mul_(m).add_(ctx_p.data, alpha=1.0 - m)

    def forward(
        self,
        images: torch.Tensor,
        ctx_indices: torch.Tensor,
        ctx_valid: torch.Tensor,
        tgt_indices: torch.Tensor,
        tgt_valid: torch.Tensor,
    ) -> torch.Tensor:
        ctx_emb = self.context_enc.forward_context(images, ctx_indices, ctx_valid)

        with torch.no_grad():
            tgt_emb_full = self.target_enc(images)
            tgt_emb = torch.gather(
                tgt_emb_full,
                1,
                tgt_indices.unsqueeze(-1).expand(-1, -1, tgt_emb_full.shape[-1]),
            )

        pred_emb = self.predictor(ctx_emb, ctx_valid, tgt_indices, tgt_valid)

        per = F.smooth_l1_loss(pred_emb, tgt_emb, reduction="none").mean(dim=-1)
        mask = tgt_valid.float()
        denom = mask.sum().clamp(min=1)
        return (per * mask).sum() / denom


In [ ]:
%%writefile src/engine/__init__.py


In [ ]:
%%writefile src/engine/pretrain.py
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.config import CONFIG
from src.data import ISICImageDataset
from src.model.jepa import IJEPA
from src.utils.masking import sample_target_block


def make_collate(n_h: int, n_w: int, scale_range, aspect_range):
    def collate(batch):
        images = torch.stack(list(batch), dim=0)
        B = images.shape[0]

        ctx_list = []
        tgt_list = []
        for _ in range(B):
            m = sample_target_block(n_h, n_w, scale_range, aspect_range)
            ctx_list.append(m["ctx_indices"])
            tgt_list.append(m["tgt_indices"])

        n_ctx_max = max(t.shape[0] for t in ctx_list)
        n_tgt_max = max(t.shape[0] for t in tgt_list)

        ctx_indices = torch.zeros(B, n_ctx_max, dtype=torch.long)
        ctx_valid = torch.zeros(B, n_ctx_max, dtype=torch.bool)
        tgt_indices = torch.zeros(B, n_tgt_max, dtype=torch.long)
        tgt_valid = torch.zeros(B, n_tgt_max, dtype=torch.bool)

        for i in range(B):
            n_c = ctx_list[i].shape[0]
            ctx_indices[i, :n_c] = ctx_list[i]
            ctx_valid[i, :n_c] = True
            n_t = tgt_list[i].shape[0]
            tgt_indices[i, :n_t] = tgt_list[i]
            tgt_valid[i, :n_t] = True

        return {
            "images": images,
            "ctx_indices": ctx_indices,
            "ctx_valid": ctx_valid,
            "tgt_indices": tgt_indices,
            "tgt_valid": tgt_valid,
        }

    return collate


def cosine_lr(step: int, total_steps: int, base_lr: float, warmup_steps: int = 100) -> float:
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    import math
    return base_lr * 0.5 * (1.0 + math.cos(progress * math.pi))


def pretrain(cfg) -> Path:
    from src.config import on_kaggle
    if on_kaggle() and cfg.device == "cpu":
        raise RuntimeError(
            "Aborting: Kaggle assigned a GPU that the installed PyTorch cannot use "
            "(sm_60 P100 or older). On the Kaggle web UI, open this notebook, "
            "Session options -> Accelerator -> GPU T4 x2, then re-run."
        )
    print(f"[pretrain] device={cfg.device}  epochs={cfg.jepa.epochs}  bs={cfg.jepa.batch_size}")
    print(f"[pretrain] data_dir={cfg.data_dir}")
    print(f"[pretrain] img={cfg.jepa.img_size} patch={cfg.jepa.patch_size} "
          f"({cfg.jepa.n_h}x{cfg.jepa.n_w}={cfg.jepa.n_h * cfg.jepa.n_w} tokens)")

    ds = ISICImageDataset(cfg.data_dir, img_size=cfg.jepa.img_size, augment=False)
    print(f"[pretrain] dataset size: {len(ds)}")

    collate = make_collate(
        cfg.jepa.n_h,
        cfg.jepa.n_w,
        tuple(cfg.jepa.mask_scale),
        tuple(cfg.jepa.mask_aspect),
    )
    dl = DataLoader(
        ds,
        batch_size=cfg.jepa.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
        collate_fn=collate,
    )

    model = IJEPA(
        img_size=cfg.jepa.img_size,
        patch_size=cfg.jepa.patch_size,
        enc_dim=cfg.jepa.enc_dim,
        enc_depth=cfg.jepa.enc_depth,
        enc_heads=cfg.jepa.enc_heads,
        pred_dim=cfg.jepa.pred_dim,
        pred_depth=cfg.jepa.pred_depth,
        pred_heads=cfg.jepa.pred_heads,
        ema_momentum=cfg.jepa.ema_momentum,
    ).to(cfg.device)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"[pretrain] params: trainable={n_trainable/1e6:.2f}M  total={n_total/1e6:.2f}M")

    optim = torch.optim.AdamW(
        [
            {"params": model.context_enc.parameters(), "lr": cfg.jepa.enc_lr},
            {"params": model.predictor.parameters(), "lr": cfg.jepa.pred_lr},
        ],
        weight_decay=cfg.jepa.weight_decay,
    )

    steps_per_epoch = max(1, len(dl))
    total_steps = max(1, steps_per_epoch * cfg.jepa.epochs)
    use_bf16 = cfg.device == "cuda"

    losses: list[float] = []
    step = 0
    t0 = time.time()

    for epoch in range(cfg.jepa.epochs):
        pbar = tqdm(dl, desc=f"epoch {epoch+1}/{cfg.jepa.epochs}", total=steps_per_epoch)
        for batch in pbar:
            images = batch["images"].to(cfg.device, non_blocking=True)
            ctx_idx = batch["ctx_indices"].to(cfg.device, non_blocking=True)
            ctx_valid = batch["ctx_valid"].to(cfg.device, non_blocking=True)
            tgt_idx = batch["tgt_indices"].to(cfg.device, non_blocking=True)
            tgt_valid = batch["tgt_valid"].to(cfg.device, non_blocking=True)

            lr = cosine_lr(step, total_steps, cfg.jepa.enc_lr, warmup_steps=min(100, total_steps // 10))
            for pg in optim.param_groups:
                pg["lr"] = lr

            optim.zero_grad(set_to_none=True)

            if use_bf16:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    loss = model(images, ctx_idx, ctx_valid, tgt_idx, tgt_valid)
            else:
                loss = model(images, ctx_idx, ctx_valid, tgt_idx, tgt_valid)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            model.update_target_encoder()

            losses.append(loss.item())
            step += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        avg = sum(losses[-steps_per_epoch:]) / steps_per_epoch
        print(f"[epoch {epoch+1}] avg_loss={avg:.4f}")

        save_every = getattr(cfg.jepa, "save_every_epochs", 10)
        if save_every and (epoch + 1) % save_every == 0:
            ckpt = cfg.output_dir / f"jepa_vit_tiny_epoch{epoch+1:03d}.pt"
            torch.save(
                {
                    "context_enc": model.context_enc.state_dict(),
                    "target_enc": model.target_enc.state_dict(),
                    "epoch": epoch + 1,
                    "avg_loss": avg,
                    "config": {
                        "img_size": cfg.jepa.img_size,
                        "patch_size": cfg.jepa.patch_size,
                        "enc_dim": cfg.jepa.enc_dim,
                        "enc_depth": cfg.jepa.enc_depth,
                        "enc_heads": cfg.jepa.enc_heads,
                    },
                },
                ckpt,
            )
            print(f"[epoch {epoch+1}] saved {ckpt}")

    elapsed = time.time() - t0
    last_n = min(50, len(losses))
    print(
        f"[pretrain] done in {elapsed:.1f}s  "
        f"first_loss={losses[0]:.4f}  last{last_n}_avg={sum(losses[-last_n:])/last_n:.4f}"
    )

    ckpt_path = cfg.output_dir / "jepa_vit_tiny_final.pt"
    torch.save(
        {
            "context_enc": model.context_enc.state_dict(),
            "target_enc": model.target_enc.state_dict(),
            "config": {
                "img_size": cfg.jepa.img_size,
                "patch_size": cfg.jepa.patch_size,
                "enc_dim": cfg.jepa.enc_dim,
                "enc_depth": cfg.jepa.enc_depth,
                "enc_heads": cfg.jepa.enc_heads,
            },
            "step": step,
            "loss_history": losses[:: max(1, len(losses) // 200)],
        },
        ckpt_path,
    )
    print(f"[pretrain] saved {ckpt_path}")
    return ckpt_path


In [ ]:
%%writefile src/train.py
"""Training entrypoint.

Run locally (CPU/MPS smoke test):   python -m src.train --mode pretrain --epochs 1
Run on Kaggle (GPU):                same command inside the notebook cell.

Keep this file as the single entrypoint the Kaggle notebook calls, so the
"run" surface never changes even as the model code grows.
"""
from __future__ import annotations

import argparse

from src.config import CONFIG, on_kaggle
from src.engine.pretrain import pretrain


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="JEPA training")
    p.add_argument("--mode", choices=["pretrain", "segment"], default="pretrain")
    p.add_argument("--epochs", type=int, default=CONFIG.jepa.epochs)
    p.add_argument("--batch-size", type=int, default=CONFIG.jepa.batch_size)
    p.add_argument("--lr", type=float, default=CONFIG.jepa.enc_lr)
    return p.parse_args()


def main() -> None:
    args = parse_args()
    print(f"[env]    running on Kaggle: {on_kaggle()}")
    print(f"[device] {CONFIG.device}")
    print(f"[paths]  data={CONFIG.data_dir}  output={CONFIG.output_dir}")
    print(f"[hparams] mode={args.mode} epochs={args.epochs} batch_size={args.batch_size} lr={args.lr}")

    if args.mode == "pretrain":
        CONFIG.jepa.epochs = args.epochs
        CONFIG.jepa.batch_size = args.batch_size
        CONFIG.jepa.enc_lr = args.lr
        CONFIG.jepa.pred_lr = args.lr
        pretrain(CONFIG)
    else:
        raise NotImplementedError("segment mode is scaffolded in the next pass")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/pretrain_jepa.yaml
img_size: 96
patch_size: 8
enc_dim: 192
enc_depth: 12
enc_heads: 3
enc_mlp_ratio: 4.0
pred_dim: 192
pred_depth: 6
pred_heads: 4
pred_mlp_ratio: 4.0
mask_scale: [0.15, 0.20]
mask_aspect: [0.75, 1.5]
ema_momentum: 0.996
enc_lr: 1.0e-3
pred_lr: 1.0e-3
weight_decay: 0.05
batch_size: 64
epochs: 50
save_every_epochs: 10


In [ ]:
import importlib
import torch

import src.config
import src.data
import src.model.vit
import src.model.predictor
import src.model.jepa
import src.engine.pretrain
importlib.reload(src.config)
importlib.reload(src.data)
importlib.reload(src.model.vit)
importlib.reload(src.model.predictor)
importlib.reload(src.model.jepa)
importlib.reload(src.engine.pretrain)
from src.config import CONFIG
from src.model.vit import ViT
from src.model.jepa import IJEPA
from src.utils.masking import sample_target_block

print('device:', CONFIG.device)
print('img:', CONFIG.jepa.img_size, 'patch:', CONFIG.jepa.patch_size,
      'tokens:', CONFIG.jepa.n_h * CONFIG.jepa.n_w)

vit = ViT(img_size=96, patch_size=8, dim=192, depth=12, heads=3).to(CONFIG.device)
x = torch.randn(2, 3, 96, 96, device=CONFIG.device)
out = vit(x)
print('ViT forward shape:', tuple(out.shape))

m = sample_target_block(12, 12)
print('mask sample: ctx=', m['n_ctx'], 'tgt=', m['n_tgt'])

model = IJEPA(
    img_size=CONFIG.jepa.img_size, patch_size=CONFIG.jepa.patch_size,
    enc_dim=CONFIG.jepa.enc_dim, enc_depth=CONFIG.jepa.enc_depth, enc_heads=CONFIG.jepa.enc_heads,
    pred_dim=CONFIG.jepa.pred_dim, pred_depth=CONFIG.jepa.pred_depth, pred_heads=CONFIG.jepa.pred_heads,
).to(CONFIG.device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'IJEPA trainable params: {n/1e6:.2f}M')


In [ ]:
!python -m src.train --mode pretrain --epochs 1
